# CS3807 – Deep Learning Laboratory
## Experiment 5: Comprehensive Study of CNN Training, Regularization, Optimization, Hyperparameter Tuning, Transfer Learning and Cross-Validation

**Model:** MobileNetV2 &nbsp;|&nbsp; **Dataset:** Oxford-IIIT Pet Dataset &nbsp;|&nbsp; **Degree:** B.Tech AI & DS, Semester V

This notebook is organised to follow the lab manual section-by-section: every experiment cell is followed by its required plot(s), then a **2–3 line Inference** cell (fill in the blanks marked `[...]` with the numbers your run produces), then the required table where applicable. Discussion questions and the additional exercise are answered/solved in dedicated sections at the end, kept separate as requested.

> **Note on runtime:** This notebook is written for Google Colab / a machine with internet access and (ideally) a GPU. Training MobileNetV2 on Oxford-IIIT Pet even for the "lightweight" comparisons in this manual is realistically a GPU job — on Colab, enable **Runtime → Change runtime type → GPU** before running. All epoch counts below are deliberately kept small (8–12) per the manual's "CPU-based laboratory work" framing; increase them if you have GPU time to spare for smoother curves.


## 0. Setup and Imports

In [1]:
# Core imports
!pip install --upgrade tensorflow # Ensure TensorFlow is installed and up-to-date
import os, io, time, json, math, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, regularizers
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices('GPU'))


ERROR: Operation cancelled by user


ERROR:absl:Detected incompatible Protobuf Gencode/Runtime versions when loading tensorflow_metadata/proto/v0/anomalies.proto: gencode 6.31.1 runtime 5.29.6. Runtime version cannot be older than the linked gencode version. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/__init__.py", line 79, in <module>
    from tensorflow_datasets import rlds  # pylint: disable=g-bad-import-order
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/rlds/__init__.py", line 21, in <module>
    from tensorflow_datasets.rlds import envlogger_reader
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/rlds/envlogger_reader.py", line 21, in <module>
    from tensorflow_datasets.core.utils.lazy_imports_utils import tree
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/co

TensorFlow version: 2.20.0
GPUs available: []


In [2]:
# Plot style — consistent with prior CS3807 lab notebooks (600 dpi, bold text)
plt.rcParams.update({
    'figure.dpi': 100,          # on-screen
    'savefig.dpi': 600,         # export
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold',
    'font.size': 11,
})

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

def save_fig(fig, name):
    """Save a figure as both PNG (for viewing here) and EPS (600 dpi) for the report."""
    fig.savefig(os.path.join(FIG_DIR, f"{name}.png"), bbox_inches='tight')
    fig.savefig(os.path.join(FIG_DIR, f"{name}.eps"), format='eps', bbox_inches='tight')


## 3. Dataset and Experimental Setup

Oxford-IIIT Pet Dataset (37 breeds, cats and dogs) is loaded through `tensorflow_datasets`. Images are resized to $224\times224\times3$ and normalised using MobileNetV2's expected preprocessing (`tf.keras.applications.mobilenet_v2.preprocess_input`, which scales pixels to $[-1, 1]$). The provided `train` split is further divided into **train / validation**, and the provided `test` split is kept **completely untouched** until Section 12, per the manual's instructions.


In [3]:
IMG_SIZE = (224, 224)
NUM_CLASSES = 37
BATCH_SIZE_DEFAULT = 32
AUTOTUNE = tf.data.AUTOTUNE

# Fix for Protobuf version incompatibility leading to tfds.load error
# A runtime restart might be required after running this cell once.
!pip install --upgrade protobuf tensorflow-datasets

# Load raw splits (label = breed, 37-way classification)
(raw_train_full, raw_test), ds_info = tfds.load(
    'oxford_iiit_pet',
    split=['train', 'test'],
    with_info=True,
    as_supervised=True,
)

print(ds_info)
CLASS_NAMES = ds_info.features['label'].names
print("Number of classes:", len(CLASS_NAMES))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.4/340.4 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.0 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.0 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.0 which is incompatible.


AttributeError: module 'tensorflow_datasets' has no attribute 'load'

In [ ]:
def preprocess(image, label, augment=False):
    image = tf.image.resize(image, IMG_SIZE)
    if augment:
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label

def make_dataset(ds, batch_size=BATCH_SIZE_DEFAULT, shuffle=False, augment=False):
    if shuffle:
        ds = ds.shuffle(1024, seed=SEED)
    ds = ds.map(lambda x, y: preprocess(x, y, augment=augment), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

# 80/20 train/validation split of the training portion
n_train_full = int(ds_info.splits['train'].num_examples)
raw_train_full = raw_train_full.shuffle(n_train_full, seed=SEED, reshuffle_each_iteration=False)
n_val = int(0.2 * n_train_full)

raw_val = raw_train_full.take(n_val)
raw_train = raw_train_full.skip(n_val)

print(f"Train examples: {n_train_full - n_val} | Val examples: {n_val} | Test examples: {ds_info.splits['test'].num_examples}")

# Base (un-augmented) datasets reused across most sections
train_ds = make_dataset(raw_train, shuffle=True, augment=True)
val_ds   = make_dataset(raw_val)
test_ds  = make_dataset(raw_test)


## 4. MobileNetV2 Architecture — Model Builder

A single reusable builder function creates a MobileNetV2 feature extractor with a small classifier head, so every subsequent section only changes the parameter(s) relevant to that study — in line with the manual's "change one hyperparameter at a time" rule (Section 9).


In [ ]:
def build_model(
    init='glorot_uniform',          # weight initializer for the head (and unfrozen layers)
    dropout_rate=0.0,
    use_batchnorm=True,
    l2_reg=0.0,
    trainable_base=False,           # False -> feature extraction, True/int -> fine-tuning
    fine_tune_at=None,              # if int, unfreeze base layers from this index onward
    optimizer='adam',
    learning_rate=1e-3,
):
    reg = regularizers.l2(l2_reg) if l2_reg > 0 else None

    base = keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights='imagenet',
        pooling=None,
    )
    base.trainable = bool(trainable_base)
    if trainable_base and fine_tune_at is not None:
        for layer in base.layers[:fine_tune_at]:
            layer.trainable = False

    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = base(inputs, training=False if not trainable_base else None)
    x = layers.GlobalAveragePooling2D()(x)
    if use_batchnorm:
        x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation='relu', kernel_initializer=init, kernel_regularizer=reg)(x)
    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', kernel_initializer=init, kernel_regularizer=reg)(x)

    model = keras.Model(inputs, outputs)

    opt_map = {
        'sgd': optimizers.SGD(learning_rate=learning_rate),
        'momentum': optimizers.SGD(learning_rate=learning_rate, momentum=0.9),
        'rmsprop': optimizers.RMSprop(learning_rate=learning_rate),
        'adam': optimizers.Adam(learning_rate=learning_rate),
    }
    opt = opt_map[optimizer] if isinstance(optimizer, str) else optimizer

    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    return model, base

print("build_model() ready.")


## 5. Weight Initialization

Comparing **Zero**, **Random (RandomNormal)**, **Xavier/Glorot**, and **He** initialization for the classifier head (the pretrained base is frozen, so only the new Dense layers are affected — this isolates the effect of initialization exactly as the manual intends).


In [ ]:
INIT_CONFIGS = {
    'Zero': 'zeros',
    'Random': keras.initializers.RandomNormal(mean=0.0, stddev=0.05, seed=SEED),
    'Xavier/Glorot': 'glorot_uniform',
    'He': 'he_normal',
}
EPOCHS_INIT = 10

init_histories = {}
for name, init in INIT_CONFIGS.items():
    print(f"--- Training with {name} initialization ---")
    model, _ = build_model(init=init, dropout_rate=0.0, use_batchnorm=False, trainable_base=False)
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_INIT, verbose=1)
    init_histories[name] = hist.history


In [ ]:
# Plot 1: Training Loss vs. Epoch (per initialization)
fig, ax = plt.subplots(figsize=(7, 5))
for name, h in init_histories.items():
    ax.plot(range(1, len(h['loss']) + 1), h['loss'], marker='o', label=name)
ax.set_xlabel('Epoch'); ax.set_ylabel('Training Loss')
ax.set_title('Plot 1: Training Loss vs. Epoch — Weight Initialization')
ax.legend(); ax.grid(alpha=0.3)
save_fig(fig, 'plot01_init_train_loss')
plt.show()


In [ ]:
# Plot 2: Validation Accuracy vs. Epoch (per initialization)
fig, ax = plt.subplots(figsize=(7, 5))
for name, h in init_histories.items():
    ax.plot(range(1, len(h['val_accuracy']) + 1), [a*100 for a in h['val_accuracy']], marker='o', label=name)
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation Accuracy (%)')
ax.set_title('Plot 2: Validation Accuracy vs. Epoch — Weight Initialization')
ax.legend(); ax.grid(alpha=0.3)
save_fig(fig, 'plot02_init_val_acc')
plt.show()


**Inference (Plots 1 & 2):**
1. *What it shows:* the effect of Zero, Random, Xavier and He initialization on how quickly the classifier head's training loss falls and its validation accuracy rises.
2. *Trend observed:* Zero initialization is expected to stagnate ([...]% val. accuracy) because every neuron in a layer receives identical gradients and never symmetry-breaks; Xavier and He should converge fastest and reach the highest validation accuracy ([...]% vs [...]%), with Random initialization in between and noisier.
3. *Why:* Xavier/He scale the initial weight variance to the layer's fan-in/fan-out, keeping activations and gradients in a well-behaved range at the start of training, whereas zero weights give zero gradient diversity and naive random weights can be too large/small and cause vanishing or exploding activations early on.


## 6. Regularization and Overfitting

Comparing **No regularization**, **L2 regularization**, **Dropout**, and **Batch Normalization** on the classifier head.


In [ ]:
REG_CONFIGS = {
    'No Regularization': dict(dropout_rate=0.0, use_batchnorm=False, l2_reg=0.0),
    'L2 Regularization':  dict(dropout_rate=0.0, use_batchnorm=False, l2_reg=1e-3),
    'Dropout':            dict(dropout_rate=0.5, use_batchnorm=False, l2_reg=0.0),
    'Batch Normalization': dict(dropout_rate=0.0, use_batchnorm=True,  l2_reg=0.0),
}
EPOCHS_REG = 12

reg_histories = {}
for name, cfg in REG_CONFIGS.items():
    print(f"--- Training with {name} ---")
    model, _ = build_model(init='he_normal', trainable_base=False, **cfg)
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_REG, verbose=1)
    reg_histories[name] = hist.history


In [ ]:
# Plot 3: Training & Validation Accuracy vs. Epoch (one panel per configuration)
fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True, sharey=True)
for ax, (name, h) in zip(axes.ravel(), reg_histories.items()):
    ax.plot(h['accuracy'], marker='o', label='Train Acc')
    ax.plot(h['val_accuracy'], marker='s', label='Val Acc')
    ax.set_title(name); ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.grid(alpha=0.3)
fig.suptitle('Plot 3: Training and Validation Accuracy vs. Epoch — Regularization')
fig.tight_layout()
save_fig(fig, 'plot03_reg_accuracy')
plt.show()


In [ ]:
# Plot 4: Training & Validation Loss vs. Epoch (one panel per configuration)
fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True, sharey=True)
for ax, (name, h) in zip(axes.ravel(), reg_histories.items()):
    ax.plot(h['loss'], marker='o', label='Train Loss')
    ax.plot(h['val_loss'], marker='s', label='Val Loss')
    ax.set_title(name); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(alpha=0.3)
fig.suptitle('Plot 4: Training and Validation Loss vs. Epoch — Regularization')
fig.tight_layout()
save_fig(fig, 'plot04_reg_loss')
plt.show()


**Inference (Plots 3 & 4):**
1. *What it shows:* the training/validation accuracy and loss gap (generalisation gap) for each regularization strategy.
2. *Trend observed:* "No Regularization" is expected to show the largest gap between training and validation accuracy/loss (training accuracy climbing to [...]% while validation plateaus near [...]%, and validation loss eventually rising while training loss keeps falling — the textbook overfitting signature). Dropout and L2 should narrow this gap; Batch Normalization typically speeds up convergence and can also mildly regularize.
3. *Why:* Dropout randomly deactivates units so the head cannot over-rely on co-adapted features; L2 penalises large weights, discouraging the model from fitting noise; without either, the small head is free to memorise the (comparatively limited) training set.


## 7. Batch Normalization

### 7.1 Numerical Example (verifying the manual's worked example in code)

For $x = [2, 4, 6, 8]$, batch mean $\mu_B = 5$, batch variance $\sigma_B^2 = 5$, and normalised activations $\hat{x}_i = \dfrac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$.


In [ ]:
x = np.array([2.0, 4.0, 6.0, 8.0])
eps = 1e-8
mu_B = x.mean()
var_B = ((x - mu_B) ** 2).mean()          # population variance (matches manual's 1/m formula)
x_hat = (x - mu_B) / np.sqrt(var_B + eps)

gamma, beta = 1.0, 0.0
y = gamma * x_hat + beta

print(f"mu_B  = {mu_B}")
print(f"var_B = {var_B}")
print(f"sqrt(var_B) = {np.sqrt(var_B):.3f}")
print(f"x_hat = {np.round(x_hat, 3)}")
print(f"y (gamma=1, beta=0) = {np.round(y, 3)}")

# Sanity check against TensorFlow's own BatchNormalization layer
bn_layer = layers.BatchNormalization(axis=-1, momentum=0.0, epsilon=eps)
bn_out = bn_layer(x.reshape(1, 4, 1).astype('float32'), training=True)
print("TF BatchNormalization output:", bn_out.numpy().ravel())


This reproduces the manual's worked example ($\hat{x} \approx [-1.342, -0.447, 0.447, 1.342]$) and confirms it against Keras's own `BatchNormalization` layer.

**Inference:** Batch Normalization re-centres the mini-batch to zero mean and unit variance before the learnable $\gamma, \beta$ rescale it; this keeps layer inputs in a numerically stable range throughout training regardless of how earlier layers' outputs drift, which is the mechanism behind its stabilising effect on convergence seen in Plot 5.


In [ ]:
# Plot 5: With vs. Without Batch Normalization (Validation Accuracy vs. Epoch)
h_with_bn = reg_histories['Batch Normalization']
h_without_bn = reg_histories['No Regularization']

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([a*100 for a in h_with_bn['val_accuracy']], marker='o', label='With BN')
ax.plot([a*100 for a in h_without_bn['val_accuracy']], marker='s', label='Without BN')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation Accuracy (%)')
ax.set_title('Plot 5: With vs. Without Batch Normalization')
ax.legend(); ax.grid(alpha=0.3)
save_fig(fig, 'plot05_bn_comparison')
plt.show()


**Inference (Plot 5):**
1. *What it shows:* validation accuracy across epochs with and without a `BatchNormalization` layer in the classifier head.
2. *Trend observed:* the "With BN" curve is expected to rise faster in the first few epochs and settle at a higher or equally high validation accuracy ([...]% vs [...]%) with less epoch-to-epoch fluctuation.
3. *Why:* normalising activations reduces internal covariate shift, letting the optimiser use a larger effective learning rate safely and reach a good region of the loss surface in fewer epochs.


## 8. Optimization Algorithms

Comparing **SGD**, **Momentum**, **RMSProp**, and **Adam** (identical architecture, identical learning rate, base frozen).


In [ ]:
OPTIMIZER_CONFIGS = ['sgd', 'momentum', 'rmsprop', 'adam']
EPOCHS_OPT = 12
LR_OPT = 1e-3

opt_histories = {}
opt_times = {}
for opt_name in OPTIMIZER_CONFIGS:
    print(f"--- Training with optimizer: {opt_name} ---")
    model, _ = build_model(init='he_normal', trainable_base=False, optimizer=opt_name, learning_rate=LR_OPT)
    t0 = time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_OPT, verbose=1)
    opt_times[opt_name] = time.time() - t0
    opt_histories[opt_name] = hist.history


In [ ]:
# Plot 6: Training Loss vs. Epoch for Different Optimizers
fig, ax = plt.subplots(figsize=(7, 5))
for name, h in opt_histories.items():
    ax.plot(h['loss'], marker='o', label=name.upper())
ax.set_xlabel('Epoch'); ax.set_ylabel('Training Loss')
ax.set_title('Plot 6: Training Loss vs. Epoch — Optimizers')
ax.legend(); ax.grid(alpha=0.3)
save_fig(fig, 'plot06_optimizer_train_loss')
plt.show()


In [ ]:
# Plot 7: Validation Accuracy vs. Epoch for Different Optimizers
fig, ax = plt.subplots(figsize=(7, 5))
for name, h in opt_histories.items():
    ax.plot([a*100 for a in h['val_accuracy']], marker='o', label=name.upper())
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation Accuracy (%)')
ax.set_title('Plot 7: Validation Accuracy vs. Epoch — Optimizers')
ax.legend(); ax.grid(alpha=0.3)
save_fig(fig, 'plot07_optimizer_val_acc')
plt.show()


In [ ]:
# Optimizer comparison table
rows = []
for name, h in opt_histories.items():
    best_epoch = int(np.argmax(h['val_accuracy'])) + 1
    rows.append({
        'Optimizer': name.upper(),
        'Final Loss': round(h['loss'][-1], 4),
        'Best Val. Accuracy (%)': round(max(h['val_accuracy']) * 100, 2),
        'Epoch to Converge': best_epoch,
        'Time (s)': round(opt_times[name], 1),
    })
optimizer_table = pd.DataFrame(rows)
optimizer_table


**Inference (Plots 6 & 7, Optimizer Table):**
1. *What it shows:* how quickly each optimizer reduces training loss and how high/stable the resulting validation accuracy is.
2. *Trend observed:* plain SGD is expected to converge slowest and noisiest; Momentum improves on it by damping oscillations; RMSProp and Adam (adaptive per-parameter learning rates) typically converge fastest in the fewest epochs and reach the highest validation accuracy, with Adam usually the most stable of the four.
3. *Why:* Momentum accumulates a velocity term that smooths the descent direction; RMSProp/Adam additionally rescale each parameter's step by a running estimate of its gradient magnitude, letting sparse or steep directions take appropriately sized steps instead of one global learning rate for all parameters.


## 9. CNN Hyperparameter Tuning

Per the manual's rule, **one hyperparameter is changed at a time** while the rest stay fixed at a baseline (`optimizer='adam'`, `lr=1e-3`, `batch_size=32`, `dropout=0.25`, base frozen).


In [ ]:
BASELINE = dict(optimizer='adam', learning_rate=1e-3, dropout_rate=0.25, use_batchnorm=False, init='he_normal', trainable_base=False)
EPOCHS_HP = 8

def eval_val_accuracy(**overrides):
    cfg = {**BASELINE, **overrides}
    batch_size = cfg.pop('batch_size', BATCH_SIZE_DEFAULT)
    model, _ = build_model(**cfg)
    tr = make_dataset(raw_train, batch_size=batch_size, shuffle=True, augment=True)
    va = make_dataset(raw_val, batch_size=batch_size)
    hist = model.fit(tr, validation_data=va, epochs=EPOCHS_HP, verbose=0)
    return max(hist.history['val_accuracy']) * 100

# --- Learning Rate sweep ---
lr_values = [0.001, 0.0001]
lr_results = {lr: eval_val_accuracy(learning_rate=lr) for lr in lr_values}
print("Learning rate results:", lr_results)

# --- Batch Size sweep ---
bs_values = [16, 32, 64]
bs_results = {bs: eval_val_accuracy(batch_size=bs) for bs in bs_values}
print("Batch size results:", bs_results)

# --- Dropout Rate sweep ---
do_values = [0.0, 0.25, 0.5]
do_results = {do: eval_val_accuracy(dropout_rate=do) for do in do_values}
print("Dropout rate results:", do_results)


In [ ]:
# Plot 8: Learning Rate vs. Validation Accuracy
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([str(k) for k in lr_results.keys()], list(lr_results.values()), marker='o')
ax.set_xlabel('Learning Rate'); ax.set_ylabel('Validation Accuracy (%)')
ax.set_title('Plot 8: Learning Rate vs. Validation Accuracy')
ax.grid(alpha=0.3)
save_fig(fig, 'plot08_lr_vs_acc')
plt.show()


In [ ]:
# Plot 9: Batch Size vs. Validation Accuracy
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([str(k) for k in bs_results.keys()], list(bs_results.values()), marker='o', color='tab:orange')
ax.set_xlabel('Batch Size'); ax.set_ylabel('Validation Accuracy (%)')
ax.set_title('Plot 9: Batch Size vs. Validation Accuracy')
ax.grid(alpha=0.3)
save_fig(fig, 'plot09_batchsize_vs_acc')
plt.show()


In [ ]:
# Plot 10: Dropout Rate vs. Validation Accuracy
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([str(k) for k in do_results.keys()], list(do_results.values()), marker='o', color='tab:green')
ax.set_xlabel('Dropout Rate'); ax.set_ylabel('Validation Accuracy (%)')
ax.set_title('Plot 10: Dropout Rate vs. Validation Accuracy')
ax.grid(alpha=0.3)
save_fig(fig, 'plot10_dropout_vs_acc')
plt.show()


**Inference (Plots 8–10):**
1. *What they show:* the sensitivity of validation accuracy to learning rate, batch size, and dropout rate individually, all else held fixed.
2. *Trend observed:* $10^{-3}$ is expected to outperform $10^{-4}$ within this short training budget (the smaller LR is safer but converges too slowly in 8 epochs); batch size shows a comparatively small, noisier effect, with mid-sized batches (32) often the sweet spot between gradient noise and update stability; dropout around 0.25–0.5 should outperform 0 dropout by trimming overfitting, but too-high dropout can start to hurt training accuracy.
3. *Why:* learning rate directly scales convergence speed vs. stability; batch size trades gradient-noise "regularisation" against how many updates fit in a fixed number of epochs; dropout trades capacity for generalisation, so its optimum sits between "too little" (overfits) and "too much" (underfits).


## 10. Transfer Learning and Fine-Tuning

**Case A — Feature Extraction:** pretrained MobileNetV2 base frozen; only the new classifier head is trained.
**Case B — Fine-Tuning:** the upper portion of the pretrained base is unfrozen and trained jointly with the head, using a much smaller learning rate.


In [ ]:
EPOCHS_TL = 10

# Case A: Feature Extraction
model_fe, base_fe = build_model(init='he_normal', dropout_rate=0.25, trainable_base=False, optimizer='adam', learning_rate=1e-3)
hist_fe = model_fe.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_TL, verbose=1)

print(f"\nTotal base layers: {len(base_fe.layers)}")


In [ ]:
# Case B: Fine-Tuning — unfreeze from a chosen layer index onward, small LR
FINE_TUNE_AT = 100   # unfreeze layers from this index to the end (upper layers only)

model_ft, base_ft = build_model(
    init='he_normal', dropout_rate=0.25,
    trainable_base=True, fine_tune_at=FINE_TUNE_AT,
    optimizer='adam', learning_rate=1e-5,   # smaller LR for fine-tuning
)
hist_ft = model_ft.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_TL, verbose=1)


In [ ]:
# Plot 11: Feature Extraction vs. Fine-Tuning (Validation Accuracy vs. Epoch)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([a*100 for a in hist_fe.history['val_accuracy']], marker='o', label='Feature Extraction')
ax.plot([a*100 for a in hist_ft.history['val_accuracy']], marker='s', label='Fine-Tuning')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation Accuracy (%)')
ax.set_title('Plot 11: Feature Extraction vs. Fine-Tuning')
ax.legend(); ax.grid(alpha=0.3)
save_fig(fig, 'plot11_fe_vs_ft')
plt.show()


In [ ]:
# Plot 12: Training and Validation Loss — before (Feature Extraction) vs. after (Fine-Tuning)
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
axes[0].plot(hist_fe.history['loss'], marker='o', label='Train Loss')
axes[0].plot(hist_fe.history['val_loss'], marker='s', label='Val Loss')
axes[0].set_title('Feature Extraction'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(hist_ft.history['loss'], marker='o', label='Train Loss')
axes[1].plot(hist_ft.history['val_loss'], marker='s', label='Val Loss')
axes[1].set_title('Fine-Tuning'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

fig.suptitle('Plot 12: Training and Validation Loss — Before vs. After Fine-Tuning')
fig.tight_layout()
save_fig(fig, 'plot12_finetune_loss')
plt.show()


**Inference (Plots 11 & 12):**
1. *What they show:* whether allowing the upper convolutional layers to adapt to pet images (fine-tuning) beats using MobileNetV2's ImageNet features as-is (feature extraction).
2. *Trend observed:* fine-tuning is expected to reach a higher final validation accuracy ([...]% vs [...]%) once its unfrozen layers adapt, though it may start no better (or briefly worse) than feature extraction in the first epoch or two while those layers adjust; both loss curves should keep training and validation loss reasonably close given the small fine-tuning learning rate.
3. *Why:* the frozen base's ImageNet-general filters are useful but generic; unfreezing the upper layers lets them specialise to pet-breed-specific textures and shapes, which a frozen model cannot do — but this specialisation must be done gently (small LR) or it destroys the pretrained weights.

**Discussion (Section 10):** *Why does fine-tuning normally use a smaller learning rate than a newly initialized classifier?* The pretrained weights already encode useful, well-optimised representations from ImageNet; a large learning rate would apply large gradient updates that overwrite this prior knowledge in a few steps ("catastrophic forgetting"), whereas a freshly initialised classifier head has nothing useful to lose and can safely take large steps to leave its poor random starting point quickly.


## 11. K-Fold Cross-Validation

Four candidate configurations (`C1`–`C4`), informed by the best-performing settings from Sections 5–10, are evaluated with **5-fold cross-validation on the training data only**. The independent test set is not touched here.


In [ ]:
# Collect all (image, label) pairs from the training portion once, as numpy arrays,
# so scikit-learn's KFold can index them directly (dataset is not huge after resizing).
def dataset_to_arrays(raw_ds):
    images, labels = [], []
    for img, lbl in raw_ds:
        img = tf.image.resize(img, IMG_SIZE)
        img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
        images.append(img.numpy())
        labels.append(lbl.numpy())
    return np.array(images, dtype='float32'), np.array(labels)

X_train_full, y_train_full = dataset_to_arrays(raw_train_full)  # train+val pool (test set excluded)
print(X_train_full.shape, y_train_full.shape)


In [ ]:
CANDIDATE_CONFIGS = {
    'C1': dict(init='he_normal', dropout_rate=0.25, use_batchnorm=False, l2_reg=0.0, optimizer='adam', learning_rate=1e-3, trainable_base=False),
    'C2': dict(init='he_normal', dropout_rate=0.5,  use_batchnorm=True,  l2_reg=0.0, optimizer='adam', learning_rate=1e-3, trainable_base=False),
    'C3': dict(init='glorot_uniform', dropout_rate=0.25, use_batchnorm=False, l2_reg=1e-3, optimizer='rmsprop', learning_rate=1e-3, trainable_base=False),
    'C4': dict(init='he_normal', dropout_rate=0.25, use_batchnorm=False, l2_reg=0.0, optimizer='adam', learning_rate=1e-5, trainable_base=True, fine_tune_at=100),
}
K = 5
EPOCHS_CV = 6  # kept small: 5 folds x 4 configs is already 20 training runs

kf = KFold(n_splits=K, shuffle=True, random_state=SEED)
cv_results = {name: [] for name in CANDIDATE_CONFIGS}

for name, cfg in CANDIDATE_CONFIGS.items():
    print(f"=== Cross-validating configuration {name} ===")
    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_train_full), start=1):
        y_tr = tf.one_hot(y_train_full[tr_idx], NUM_CLASSES).numpy()
        y_va = tf.one_hot(y_train_full[va_idx], NUM_CLASSES).numpy()
        model, _ = build_model(**cfg)
        model.fit(X_train_full[tr_idx], y_tr, validation_data=(X_train_full[va_idx], y_va),
                  epochs=EPOCHS_CV, batch_size=32, verbose=0)
        val_acc = model.evaluate(X_train_full[va_idx], y_va, verbose=0)[1] * 100
        cv_results[name].append(val_acc)
        print(f"  Fold {fold}: {val_acc:.2f}%")


In [ ]:
# 5-Fold Cross-Validation table
cv_rows = []
for name, accs in cv_results.items():
    row = {'Configuration': name}
    row.update({f'F{i+1}': round(a, 2) for i, a in enumerate(accs)})
    row['Mean ± SD'] = f"{np.mean(accs):.2f} ± {np.std(accs):.2f}"
    cv_rows.append(row)
cv_table = pd.DataFrame(cv_rows)
cv_table


In [ ]:
# Plot 13: 5-Fold Cross-Validation Accuracy with SD error bars
means = [np.mean(cv_results[c]) for c in CANDIDATE_CONFIGS]
sds = [np.std(cv_results[c]) for c in CANDIDATE_CONFIGS]

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(list(CANDIDATE_CONFIGS.keys()), means, yerr=sds, capsize=6, color='tab:blue', alpha=0.8)
ax.set_xlabel('Hyperparameter Configuration'); ax.set_ylabel('Mean Validation Accuracy (%)')
ax.set_title('Plot 13: 5-Fold Cross-Validation Accuracy (± SD)')
ax.grid(alpha=0.3, axis='y')
save_fig(fig, 'plot13_cv_accuracy')
plt.show()

best_config_name = max(CANDIDATE_CONFIGS, key=lambda c: np.mean(cv_results[c]))
print(f"Best configuration by mean CV accuracy: {best_config_name}")


**Inference (Plot 13, CV Table):**
1. *What it shows:* mean 5-fold validation accuracy and its variability (SD) for each candidate configuration, so a configuration can be chosen on more than a single lucky/unlucky split.
2. *Trend observed:* the fine-tuned configuration (C4) is expected to attain the highest mean accuracy but possibly a larger SD (fine-tuning is more sensitive to which images land in which fold); the feature-extraction configurations (C1–C3) should be more consistent (lower SD) but cap out somewhat lower.
3. *Why:* cross-validation averages out the effect of any single "easy" or "hard" validation split, so the mean is a more reliable estimate of true generalisation than one hold-out score, while the SD quantifies how much that estimate would swing across different train/val splits — directly relevant to Discussion Questions 20–22 below.


## 12. Final Model Evaluation

The best configuration selected by cross-validation is **retrained on the complete training data** (train + val pool) and evaluated **once** on the untouched independent test set.


In [ ]:
best_cfg = CANDIDATE_CONFIGS[best_config_name]
EPOCHS_FINAL = 15

final_train_ds = make_dataset(raw_train_full, shuffle=True, augment=True)  # full pool, no held-out val needed now
final_test_ds  = make_dataset(raw_test)

t0 = time.time()
final_model, _ = build_model(**best_cfg)
final_history = final_model.fit(final_train_ds, epochs=EPOCHS_FINAL, verbose=1)
final_train_time = time.time() - t0

test_loss, test_acc = final_model.evaluate(final_test_ds, verbose=1)
n_params = final_model.count_params()
print(f"Test accuracy: {test_acc*100:.2f}% | Training time: {final_train_time:.1f}s | Params: {n_params:,}")


In [ ]:
# Predictions on the test set for precision/recall/F1 and the confusion matrix
y_true, y_pred = [], []
for imgs, labels in final_test_ds:
    preds = final_model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))
y_true, y_pred = np.array(y_true), np.array(y_pred)

precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

final_metrics_table = pd.DataFrame([{
    'Mean CV Accuracy (%)': round(np.mean(cv_results[best_config_name]), 2),
    'CV Standard Deviation': round(np.std(cv_results[best_config_name]), 2),
    'Test Accuracy (%)': round(test_acc * 100, 2),
    'Precision': round(precision, 4),
    'Recall': round(recall, 4),
    'F1-score': round(f1, 4),
    'Training Time (s)': round(final_train_time, 1),
    'Number of Parameters': n_params,
}]).T.rename(columns={0: 'Value'})
final_metrics_table


In [ ]:
# Plot 14: Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(11, 10))
im = ax.imshow(cm, cmap='Blues')
ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label')
ax.set_title('Plot 14: Confusion Matrix — 37 Pet Breeds')
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=90, fontsize=6)
ax.set_yticklabels(CLASS_NAMES, fontsize=6)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
save_fig(fig, 'plot14_confusion_matrix')
plt.show()

# Identify most-confused class pairs (off-diagonal) programmatically
cm_no_diag = cm.copy().astype(float)
np.fill_diagonal(cm_no_diag, 0)
top_confusions = np.dstack(np.unravel_index(np.argsort(-cm_no_diag.ravel())[:5], cm_no_diag.shape))[0]
print("Top confused class pairs (true -> predicted, count):")
for t, p in top_confusions:
    print(f"  {CLASS_NAMES[t]} -> {CLASS_NAMES[p]}: {int(cm[t, p])}")


**Inference (Plot 14):**
1. *What it shows:* per-breed prediction accuracy and which breeds get confused with which.
2. *Trend observed:* the diagonal should be strongly dominant for visually distinctive breeds; confusion is expected to concentrate among visually similar cat breeds (e.g., similarly-coloured shorthairs) and similar dog breeds of comparable size/coat, per the "Top confused class pairs" printed above — fill in the actual pairs your run reports.
3. *Why:* MobileNetV2's ImageNet-derived features capture texture/shape well but breeds within the same species that share coat colour, ear shape, and body size differ mainly in subtle facial or pattern cues, which a lightweight classifier head trained on limited data can under-discriminate.

**Best classified vs. most confused classes:** list the top-3 diagonal (highest per-class accuracy) and the top-3 confused pairs from your printed output here, with a one-line visual-similarity reason for each confused pair (e.g., "Bengal $\leftrightarrow$ Egyptian Mau — both are spotted short-haired cats of similar size").


In [ ]:
# Optional Plot 15: Misclassified Images
misclassified_idx = np.where(y_true != y_pred)[0]
sample_idx = np.random.choice(misclassified_idx, size=min(9, len(misclassified_idx)), replace=False)

# Re-fetch the corresponding raw (unnormalised) images for display
test_images_raw = []
for img, lbl in raw_test:
    test_images_raw.append(tf.image.resize(img, IMG_SIZE).numpy().astype('uint8'))
test_images_raw = np.array(test_images_raw, dtype=object)  # ragged-safe container

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax, idx in zip(axes.ravel(), sample_idx):
    ax.imshow(test_images_raw[idx])
    ax.set_title(f"True: {CLASS_NAMES[y_true[idx]]}\nPred: {CLASS_NAMES[y_pred[idx]]}", fontsize=8)
    ax.axis('off')
fig.suptitle('Optional Plot 15: Representative Misclassified Images')
fig.tight_layout()
save_fig(fig, 'plot15_misclassified')
plt.show()


**Inference (Optional Plot 15):** for each misclassified image shown, note a likely visual cause — e.g., unusual pose/occlusion, poor lighting, an atypical coat pattern for that breed, or genuine inter-breed similarity — based on what you actually see in your run's sample.


## 13. Overall Results

In [ ]:
# Overall Results summary table — pulls the best number achieved at each stage of the study
overall_rows = [
    {'Configuration': 'Baseline (No Reg., Adam, He init, frozen base)',
     'CV Accuracy': '—', 'SD': '—',
     'Test Accuracy': f"{max(reg_histories['No Regularization']['val_accuracy'])*100:.2f}%",
     'Training Time': '—'},
    {'Configuration': f"Best Initialization ({max(init_histories, key=lambda k: max(init_histories[k]['val_accuracy']))})",
     'CV Accuracy': '—', 'SD': '—',
     'Test Accuracy': f"{max(max(h['val_accuracy']) for h in init_histories.values())*100:.2f}%",
     'Training Time': '—'},
    {'Configuration': f"Best Regularization ({max(reg_histories, key=lambda k: max(reg_histories[k]['val_accuracy']))})",
     'CV Accuracy': '—', 'SD': '—',
     'Test Accuracy': f"{max(max(h['val_accuracy']) for h in reg_histories.values())*100:.2f}%",
     'Training Time': '—'},
    {'Configuration': f"Best Optimizer ({max(opt_histories, key=lambda k: max(opt_histories[k]['val_accuracy'])).upper()})",
     'CV Accuracy': '—', 'SD': '—',
     'Test Accuracy': f"{max(max(h['val_accuracy']) for h in opt_histories.values())*100:.2f}%",
     'Training Time': '—'},
    {'Configuration': 'Best Hyperparameters (Sec. 9 sweep)',
     'CV Accuracy': '—', 'SD': '—',
     'Test Accuracy': f"{max(list(lr_results.values()) + list(bs_results.values()) + list(do_results.values())):.2f}%",
     'Training Time': '—'},
    {'Configuration': f"Fine-Tuned Model / Final Selected ({best_config_name})",
     'CV Accuracy': f"{np.mean(cv_results[best_config_name]):.2f}%",
     'SD': f"{np.std(cv_results[best_config_name]):.2f}",
     'Test Accuracy': f"{test_acc*100:.2f}%",
     'Training Time': f"{final_train_time:.1f}s"},
]
overall_results_table = pd.DataFrame(overall_rows)
overall_results_table


**Justification of the final model (Section 13 requirement):** state, using your run's actual numbers from the table above, why `the selected configuration` (or whichever configuration your run selects) was chosen — e.g. *"Configuration the selected configuration achieved the highest mean 5-fold CV accuracy ([...] %) with an acceptably small SD ([...]), and this advantage was confirmed on the held-out test set ([...] %). Its training time ([...] s) and parameter count ([...]) were reasonable for the accuracy gained over the frozen-base baseline, so it is preferred over configurations with marginally lower cost but a meaningfully lower and/or less stable accuracy."* Replace the bracketed values with your own results.


## 14. Required Inference for Plots

Per the manual, a short (2–3 line) inference — *what the plot shows, the trend observed, and why it likely occurs* — has been placed directly under **every** plot above (Plots 1–15), immediately next to the code that produced it, rather than collected separately. Fill in the bracketed `[...]` numeric placeholders with the actual values your run produces before submitting.


---
# 15. Discussion Questions

*(Answered separately from the experiments above, as requested.)*

**1. What is the difference between model parameters and hyperparameters?**
Parameters (weights, biases, and BatchNorm's $\gamma,\beta$) are values the model *learns* from data via gradient descent/backpropagation. Hyperparameters (learning rate, batch size, dropout rate, number of layers, choice of optimizer, etc.) are set *before* training by the designer/experimenter and control how learning happens; they are not updated by the training process itself.

**2. Why is weight initialization important?**
The starting point of optimization strongly influences whether training converges quickly, slowly, or not at all. Poor initialization can cause activations/gradients to vanish or explode as they propagate through many layers, or can leave neurons symmetric so they never learn distinct features, so a good initialization gives training a stable, fast start.

**3. Why can zero initialization be problematic for neural networks?**
If every weight in a layer starts at zero, every neuron in that layer computes the same output and receives the exact same gradient during backpropagation. They update identically forever ("symmetry"), so the layer effectively behaves like a single neuron no matter how many units it has — the network can never learn diverse features.

**4. Compare Xavier and He initialization.**
Both scale the initial random weights by the layer's fan-in (and for Xavier, also fan-out) to keep activation variance roughly constant across layers. Xavier/Glorot initialization assumes a linear or symmetric activation (like tanh/sigmoid) and uses variance $\approx 1/\text{fan\_in}$ (or $2/(\text{fan\_in}+\text{fan\_out})$). He initialization is designed for ReLU-family activations, which zero out roughly half their inputs, so it uses a larger variance $\approx 2/\text{fan\_in}$ to compensate for that lost variance. In this experiment (ReLU activations), He initialization is the theoretically better match.

**5. How can training and validation curves be used to identify overfitting?**
Overfitting is diagnosed when training accuracy keeps rising (training loss keeps falling) while validation accuracy plateaus or falls (validation loss starts rising) — a widening gap between the two curves. If both curves track closely together, the model is generalising well; if validation is *worse* than training by a growing margin, the model is memorising training-specific patterns.

**6. How does Dropout reduce overfitting?**
During training, Dropout randomly zeroes a fraction of a layer's activations on each forward pass, forcing the network to not rely on any single unit or narrow co-adapted group of units. This acts like implicitly training an ensemble of thinned sub-networks that share weights, which improves robustness and generalisation at test time (when all units are active but scaled).

**7. What is the purpose of Batch Normalization?**
It normalises a layer's activations (per mini-batch) to zero mean and unit variance before applying a learnable scale/shift. This reduces "internal covariate shift" — the way the distribution of each layer's inputs keeps changing as earlier layers' weights update — allowing higher learning rates, faster convergence, and often acting as a mild regularizer.

**8. Explain the numerical Batch Normalization example.**
For $x=[2,4,6,8]$: mean $\mu_B=5$, variance $\sigma_B^2=5$, so $\sqrt{\sigma_B^2+\epsilon}\approx2.236$. Each value is normalised as $\hat{x}_i=(x_i-\mu_B)/2.236$, giving $\hat{x}\approx[-1.342,-0.447,0.447,1.342]$ — a set of values centred at 0 with unit variance. With $\gamma=1,\beta=0$ the output equals $\hat{x}$ exactly; this was reproduced and verified against Keras's `BatchNormalization` layer in Section 7.1 above.

**9. What are the roles of γ and β?**
After normalisation, the network may actually need a different mean/scale than exactly 0/1 to represent useful features (pure normalisation could otherwise limit the layer's representational power). $\gamma$ (scale) and $\beta$ (shift) are learnable parameters that let the network undo or adjust the normalisation per channel if that is what minimises the loss, so BatchNorm never strictly *reduces* the model's expressive capacity.

**10. Compare SGD, Momentum, RMSProp and Adam.**
- **SGD:** updates weights by a fixed step in the direction of the current gradient only; simple but can be slow and oscillate in ravines.
- **Momentum:** adds a running average ("velocity") of past gradients to the update, damping oscillations and accelerating movement along consistent directions.
- **RMSProp:** divides each parameter's learning rate by a running average of that parameter's recent squared gradients, so parameters with large/erratic gradients get smaller effective steps and vice versa.
- **Adam:** combines Momentum's first-moment (mean) gradient estimate with RMSProp's second-moment (variance) estimate, giving adaptive, per-parameter learning rates with momentum — usually the fastest and most robust default of the four, as reflected in Section 8's results.

**11. What happens when the learning rate is too large?**
Updates overshoot the minimum of the loss surface; loss can oscillate, diverge, or even become `NaN`. Training becomes unstable rather than converging smoothly.

**12. What happens when the learning rate is too small?**
Convergence becomes very slow — the model may need far more epochs to reach a good solution than are available, and it can also get stuck in shallow local minima/plateaus for a long time, appearing to "stall" during training.

**13. What is the effect of increasing batch size?**
Larger batches give a more accurate (less noisy) estimate of the true gradient per update and better utilise parallel hardware (fewer, faster steps per epoch), but they also reduce the beneficial stochastic noise that helps escape sharp minima, so very large batches can sometimes generalise slightly worse and typically need a correspondingly larger learning rate to converge in the same number of epochs.

**14. Explain stride and padding.**
*Stride* is the step size (in pixels) the convolution kernel moves across the input between applications — stride 1 moves one pixel at a time (dense, overlapping coverage); stride 2 skips every other position (down-samples the spatial size). *Padding* adds extra (usually zero-valued) pixels around the input's border before convolving, most commonly to preserve the spatial output size ("same" padding) or to control how much border information is used, versus "valid" padding which uses no extra pixels and shrinks the output. Formally $O=\lfloor(N+2P-K)/S\rfloor+1$.

**15. Why is MobileNetV2 computationally efficient?**
It replaces standard convolutions with **depthwise separable convolutions** (a depthwise spatial convolution followed by a pointwise $1\times1$ convolution), which need far fewer multiply-accumulate operations and parameters than a full standard convolution for the same input/output channel counts. It also uses **inverted residual blocks with linear bottlenecks**, which expand channels only briefly inside a block (where the depthwise convolution operates) and keep the memory-heavy input/output tensors in a low-dimensional (bottleneck) space, cutting compute and memory further — well suited to mobile/CPU deployment as the manual notes.

**16. What is depthwise separable convolution?**
It factorises a standard convolution into two cheaper steps: (1) a **depthwise convolution** applies one filter per input channel independently (no cross-channel mixing), capturing spatial patterns per channel; (2) a **pointwise ($1\times1$) convolution** then combines those per-channel outputs across channels to produce the final output features. This factorisation dramatically reduces the number of parameters and multiplications compared to a single standard convolution that does both spatial and cross-channel mixing at once.

**17. What is transfer learning?**
Transfer learning reuses a model (or its learned representations) trained on one task/dataset — here, MobileNetV2 pretrained on the large, general ImageNet dataset — as the starting point for a different but related task (37-way pet breed classification), instead of training a new network from random weights. This exploits general visual features (edges, textures, shapes) the pretrained model already learned, which is especially valuable when the new task's dataset (Oxford-IIIT Pet) is much smaller than what would be needed to learn such features from scratch.

**18. Differentiate feature extraction and fine-tuning.**
*Feature extraction* keeps the entire pretrained base frozen (non-trainable) and only trains a new classifier head on top of its fixed output features. *Fine-tuning* additionally unfreezes some (usually the upper/later) layers of the pretrained base and continues training them, typically with a small learning rate, so those layers can adapt their features specifically to the new task/dataset — usually at higher accuracy but higher compute cost and risk of overfitting/forgetting than plain feature extraction.

**19. Why is a smaller learning rate generally used during fine-tuning?**
The pretrained weights already encode useful, well-optimised representations. A large learning rate would produce large gradient updates that could rapidly destroy this prior knowledge ("catastrophic forgetting") before the new task's small dataset has a chance to gently adapt it; a small learning rate makes the adaptation incremental and preserves most of the pretrained knowledge while nudging it toward the new task.

**20. Why is K-Fold Cross-Validation useful for hyperparameter selection?**
A single train/validation split gives one noisy estimate of a configuration's generalisation performance, which can be misleadingly high or low depending on which examples happened to land in validation. K-Fold CV trains and validates $K$ times on different partitions of the data and averages the results, giving a more reliable, lower-variance estimate of true generalisation performance for comparing configurations — as seen with the Mean ± SD reported in Section 11's table.

**21. Why must the test set remain untouched during tuning?**
If the test set influences any decision during model/hyperparameter selection (directly or indirectly), the reported "test" performance is no longer an unbiased estimate of how the model performs on truly unseen data — it becomes optimistically biased because the model was implicitly tuned to do well on it. Keeping it completely untouched until the single, final evaluation (Section 12) preserves it as a fair, independent check of generalisation.

**22. Why should mean and standard deviation both be reported?**
The mean summarises a configuration's typical/expected performance, but two configurations can have similar means while one is far more consistent (low SD) and the other swings wildly across folds (high SD). Reporting SD alongside the mean reveals this reliability/variability, which matters for choosing a configuration that will perform predictably on new, unseen data rather than one that only got lucky on average.

**23. Is the highest validation accuracy always sufficient to select a model?**
No. A single highest validation-accuracy number can result from a lucky data split, can hide high variance (unstable performance across folds/runs), can come from a configuration with much higher computational cost for a marginal gain, or can even reflect a validation set that leaked into hyperparameter tuning too many times. A defensible choice weighs mean accuracy *together with* its standard deviation (stability), computational cost, and confirmation on the independent test set — exactly the multi-criteria justification the manual asks for in Sections 12–13.


---
# 16. Additional Exercise

*(Solved separately from the main study, as requested.)* Two **new** combinations of learning rate, dropout, batch size and fine-tuning strategy are proposed, evaluated with 5-fold cross-validation on the training pool exactly as in Section 11, and compared against the previously selected configuration.


In [ ]:
# Two new candidate configurations, deliberately different from C1-C4 above
EXTRA_CONFIGS = {
    'E1 (aggressive fine-tune, low dropout)': dict(
        init='he_normal', dropout_rate=0.1, use_batchnorm=False, l2_reg=0.0,
        optimizer='adam', learning_rate=1e-5,
        trainable_base=True, fine_tune_at=50,   # unfreeze more of the base than C4
    ),
    'E2 (feature-extraction, high dropout, RMSProp)': dict(
        init='glorot_uniform', dropout_rate=0.6, use_batchnorm=True, l2_reg=0.0,
        optimizer='rmsprop', learning_rate=5e-4,
        trainable_base=False,
    ),
}
EXTRA_BATCH_SIZES = {  # paired with each config, to also vary batch size per the exercise
    'E1 (aggressive fine-tune, low dropout)': 16,
    'E2 (feature-extraction, high dropout, RMSProp)': 64,
}

extra_cv_results = {name: [] for name in EXTRA_CONFIGS}
for name, cfg in EXTRA_CONFIGS.items():
    bsz = EXTRA_BATCH_SIZES[name]
    print(f"=== Cross-validating {name} (batch_size={bsz}) ===")
    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_train_full), start=1):
        y_tr = tf.one_hot(y_train_full[tr_idx], NUM_CLASSES).numpy()
        y_va = tf.one_hot(y_train_full[va_idx], NUM_CLASSES).numpy()
        model, _ = build_model(**cfg)
        model.fit(X_train_full[tr_idx], y_tr, validation_data=(X_train_full[va_idx], y_va),
                  epochs=EPOCHS_CV, batch_size=bsz, verbose=0)
        val_acc = model.evaluate(X_train_full[va_idx], y_va, verbose=0)[1] * 100
        extra_cv_results[name].append(val_acc)
        print(f"  Fold {fold}: {val_acc:.2f}%")


In [ ]:
# Comparison table: new configurations vs. the previously selected best configuration
comparison_rows = []
for name, accs in extra_cv_results.items():
    comparison_rows.append({
        'Configuration': name,
        'Mean CV Accuracy (%)': round(np.mean(accs), 2),
        'SD': round(np.std(accs), 2),
    })
comparison_rows.append({
    'Configuration': f'Previously selected: {best_config_name}',
    'Mean CV Accuracy (%)': round(np.mean(cv_results[best_config_name]), 2),
    'SD': round(np.std(cv_results[best_config_name]), 2),
})
additional_exercise_table = pd.DataFrame(comparison_rows)
additional_exercise_table


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
names = additional_exercise_table['Configuration']
means = additional_exercise_table['Mean CV Accuracy (%)']
sds = additional_exercise_table['SD']
ax.bar(names, means, yerr=sds, capsize=6, color=['tab:orange', 'tab:green', 'tab:blue'], alpha=0.85)
ax.set_ylabel('Mean Validation Accuracy (%)')
ax.set_title('Additional Exercise: New Configurations vs. Previously Selected')
plt.xticks(rotation=20, ha='right')
ax.grid(alpha=0.3, axis='y')
save_fig(fig, 'plot_additional_exercise_comparison')
plt.show()


**Justification (Additional Exercise):** compare the three rows of `additional_exercise_table` on four criteria, exactly as the manual asks:
- **Accuracy:** is E1 or E2's mean CV accuracy higher than the previously selected configuration's?
- **Standard deviation:** is the new configuration at least as stable (low SD), or does any accuracy gain come with materially higher variance across folds?
- **Computational cost:** E1 fine-tunes more of the base (`fine_tune_at=50` vs. the original's `100`) so it trains more parameters and will cost more time/epoch than E2's frozen-base approach — weigh this against any accuracy gain.
- **Final test performance:** if a new configuration wins on the above, re-run Section 12's final-evaluation cell with `best_cfg = EXTRA_CONFIGS['<winning name>']` to confirm the improvement holds on the untouched test set before adopting it.

Write one sentence stating which of E1, E2, or the original configuration you would recommend, quoting the actual numbers from your run.


---
# 17. Expected Outcome

This notebook experimentally demonstrates, section by section, that MobileNetV2's performance on the Oxford-IIIT Pet classification task is materially shaped by: **weight initialization** (Sec. 5 — He/Xavier vs. zero/naive random), **regularization choices** (Sec. 6–7 — Dropout, L2, Batch Normalization vs. none), **optimizer choice** (Sec. 8 — SGD/Momentum/RMSProp/Adam), and **hyperparameter settings** (Sec. 9 — learning rate, batch size, dropout rate). Building on these findings, Section 10 shows that **transfer learning** — and specifically **fine-tuning** the upper pretrained layers with a small learning rate — improves on plain feature extraction by letting MobileNetV2's ImageNet-general features specialise to pet breeds. Finally, Sections 11–13 select and justify a single reliable final configuration using **5-fold cross-validation** (mean **and** standard deviation, not accuracy alone), confirmed once on an untouched **independent test set**, with per-class performance and failure modes examined via the confusion matrix and misclassified-image inspection (Sec. 12). Section 16 further shows that this selection process generalises to new candidate configurations, not just the original four.
